# Segment and Cost an Offer

**The business question.** Marlow & Finch wants to stop sending every customer the same offer. Your job: segment the customers, design a different offer for each segment, say roughly what each would cost and earn, and recommend **one thing to launch, change or stop**. You will finish with a short note to the marketing director.

**What is different this time.** You cluster on five columns instead of four. The extra column, `CategoryBreadth`, counts how many different product categories a customer has bought from. A customer who only ever buys kitchenware and one who buys from all five categories are different customers, even if they spend the same.

**About the data.** The Marlow & Finch tables are **synthetic**: a fictional retailer, generated for teaching. The costs and uplifts you will use for the offers are **invented by you**. They are assumptions, not findings, and your note must say so.

**How this notebook works.** One part, five steps:

1. Load and look
2. Run the model
3. Read the output in plain English
4. Check against JMP
5. Say it to a manager (and cost the offers)

Copilot is switched on. Check what it gives you against JMP. There is a Stretch space at the end.

## Set up

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

N_INIT = 25                      # k-means tries 25 different starting points and keeps the best one
SEED = 1                         # fixes the random choices so a re-run gives the same answer
REFERENCE_DATE = "2025-07-01"    # recency is counted in days up to this date
# Scaling: StandardScaler on the clustering columns, always.
print("Ready.")

---
## Step 1. Load and look

In [ ]:
customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")

query = """
SELECT c.CustomerID,
       CAST(julianday(:ref) - julianday(MAX(o.OrderDate)) AS INTEGER) AS RecencyDays,
       COUNT(o.OrderID)            AS Orders,
       AVG(o.OrderValue)           AS AvgOrder,
       AVG(o.Discounted)           AS DiscountShare,
       COUNT(DISTINCT o.Category)  AS CategoryBreadth,
       SUM(o.OrderValue)           AS TotalSpend
FROM customers AS c
LEFT JOIN orders AS o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID
"""
con = sqlite3.connect(":memory:")
customers.to_sql("customers", con, index=False)
orders.to_sql("orders", con, index=False)
features = pd.read_sql(query, con, params={"ref": REFERENCE_DATE})

never = features["RecencyDays"].isna().sum()
buyers = features.dropna(subset=["RecencyDays"]).reset_index(drop=True)
print(f"{len(features):,} customers, of whom {never} have never ordered. We cluster the {len(buyers):,} buyers.")
print()
print(buyers.drop(columns="CustomerID").describe().round(2).to_string())

**What to look for.** `CategoryBreadth` runs from 1 to 5. The customers who never ordered are set aside, so remember to say something about them in your note.

## Step 2. Run the model

In [ ]:
FEATURES = ["RecencyDays", "Orders", "AvgOrder", "DiscountShare", "CategoryBreadth"]
scaled = StandardScaler().fit_transform(buyers[FEATURES])       # put every column on the same scale

rows = []
for k in range(2, 9):
    km_k = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(scaled)
    rows.append({"k": k, "inertia": km_k.inertia_, "silhouette": silhouette_score(scaled, km_k.labels_)})
sweep = pd.DataFrame(rows)
print(sweep.round({"inertia": 1, "silhouette": 3}).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(sweep["k"], sweep["inertia"], marker="o")
ax[0].set_title("Inertia")
ax[1].plot(sweep["k"], sweep["silhouette"], marker="o", color="tab:orange")
ax[1].set_title("Silhouette")
for a in ax:
    a.set_xlabel("k, the number of groups")
plt.tight_layout()
plt.show()

**Choose k and justify it.** The numbers narrow the choice. Business usefulness decides it: how many different offers could the marketing team realistically run? Set `K_CHOSEN` below, and write your reason in the cell after it.

In [ ]:
K_CHOSEN = 4

km = KMeans(n_clusters=K_CHOSEN, n_init=N_INIT, random_state=SEED).fit(scaled)
buyers["Group"] = km.labels_
print("Customers in each group:", buyers["Group"].value_counts().sort_index().tolist())
print("Silhouette at this k:", round(silhouette_score(scaled, km.labels_), 3))

**Your reason for this k:**

## Step 3. Read the output in plain English

In [ ]:
profile = buyers.groupby("Group").agg(
    Customers=("CustomerID", "size"),
    RecencyDays=("RecencyDays", "mean"),
    Orders=("Orders", "mean"),
    AvgOrder=("AvgOrder", "mean"),
    DiscountShare=("DiscountShare", "mean"),
    CategoryBreadth=("CategoryBreadth", "mean"),
    TotalSpend=("TotalSpend", "mean"),
)
profile["ShareOfCustomers"] = profile["Customers"] / profile["Customers"].sum() * 100
revenue = buyers.groupby("Group")["TotalSpend"].sum()
profile["ShareOfRevenue"] = revenue / revenue.sum() * 100
print(profile.round(2).to_string())

**Name your groups.** Fill in the dictionary below with a short name for each group number, based on the profile table. Names such as "frequent small buyers" help; group numbers do not.

In [ ]:
NAMES = {}          # for example {0: "quiet spenders", 1: "frequent small buyers"}; one entry per group number

def label(g):
    return NAMES.get(g, f"Group {g}")

print({g: label(g) for g in profile.index})

## Step 4. Check against JMP

Export the table, run K Means in JMP with the same k, and compare group sizes. The settings are in the Activity Workbook, Clustering in JMP. The labels will not match and the sizes may differ slightly, which is expected.

In [ ]:
Path("exports").mkdir(exist_ok=True)
export = buyers.assign(CustomerNumber=np.arange(1, len(buyers) + 1))
export = export[["CustomerNumber", "CustomerID"] + FEATURES + ["TotalSpend"]]
export.to_csv("exports/marlow_finch_five_feature_table.csv", index=False)

jmp_sizes = []          # type the group sizes JMP gives you, largest first

python_sizes = sorted(buyers["Group"].value_counts().tolist(), reverse=True)
if len(jmp_sizes) == K_CHOSEN:
    check = pd.DataFrame({"Python": python_sizes, "JMP": jmp_sizes})
    check["Difference"] = check["Python"] - check["JMP"]
    print(check.to_string(index=False))
else:
    print("Saved exports/marlow_finch_five_feature_table.csv.")
    print("Python group sizes, largest first:", python_sizes)
    print("Add the JMP sizes to jmp_sizes above and run this cell again.")

In [ ]:
ward = AgglomerativeClustering(n_clusters=K_CHOSEN, linkage="ward").fit_predict(scaled)
print("Ward group sizes, largest first:", sorted(pd.Series(ward).value_counts().tolist(), reverse=True))
print("Adjusted Rand index, k-means against Ward:", round(adjusted_rand_score(buyers["Group"], ward), 3))

## Step 5. Say it to a manager

### Cost the offers

First, a baseline. Suppose the retailer keeps sending **one blanket offer to every buyer**. The two numbers below are **invented** assumptions: what the offer costs per customer, and how much extra a customer spends as a share of what they already spend.

In [ ]:
BLANKET_COST = 5.0       # invented: cost of the offer per customer, in pounds
BLANKET_UPLIFT = 0.03    # invented: extra spend as a share of the customer's current spend

blanket = buyers.groupby("Group").agg(Customers=("CustomerID", "size"), MeanSpend=("TotalSpend", "mean"))
blanket["NetBenefit"] = blanket["Customers"] * (blanket["MeanSpend"] * BLANKET_UPLIFT - BLANKET_COST)
print(blanket.round(0).to_string())
print()
print(f"Blanket offer, net benefit across all {len(buyers):,} buyers: {blanket['NetBenefit'].sum():,.0f} pounds")

Now **design a different offer for each group**. Fill in `TARGETED` below with one entry per group number: `(cost per customer, uplift as a share of current spend)`. Both are your own invented assumptions, so pick values you can defend in one sentence each. An offer that costs more than it earns is allowed: it tells you what to stop.

In [ ]:
TARGETED = {}       # for example {0: (3.0, 0.02), 1: (2.0, 0.03)}; one (cost, uplift) pair per group number

if len(TARGETED) == K_CHOSEN:
    offers = profile[["Customers", "TotalSpend"]].copy()
    offers.insert(0, "Name", [label(g) for g in offers.index])
    offers["Cost"] = [TARGETED[g][0] for g in offers.index]
    offers["Uplift"] = [TARGETED[g][1] for g in offers.index]
    offers["NetBenefit"] = offers["Customers"] * (offers["TotalSpend"] * offers["Uplift"] - offers["Cost"])
    print(offers.round(2).to_string())
    print()
    print(f"Targeted offers, net benefit: {offers['NetBenefit'].sum():,.0f} pounds")
    print(f"Blanket offer, net benefit:   {blanket['NetBenefit'].sum():,.0f} pounds")
else:
    print("Add one (cost, uplift) pair per group to TARGETED above, then run this cell again.")

### Write the note

Write a note to the marketing director in **150 words or fewer**. It must:

- name your segments and say how many customers and how much revenue each holds
- give one offer for each segment, with its invented cost
- recommend one thing to **launch, change or stop**
- say honestly which numbers are invented and what you would test first
- mention the customers who have never ordered

Paste your note between the quotes below. The cell counts the words and saves the note as `manager_note.md` in this folder, ready to commit to your repository.

In [ ]:
note = """

"""
words = len(note.split())
if note.strip():
    print(f"{words} words" + ("" if words <= 150 else ", which is over the 150-word limit"))
    Path("manager_note.md").write_text(note.strip() + "\n", encoding="utf-8")
    print("Saved manager_note.md")
else:
    print("Paste your note between the quotes above, then run this cell again.")

**Ask Copilot.** Try:

> Here is my note to a marketing director: [paste your note]. Which of my claims are not supported by the table above, and which numbers are assumptions I have not flagged as invented?

Then edit your note. Copilot is a good critic here, but you still decide what the data supports.

---
# Stretch

Quick finishers: use Copilot to write the code and check the numbers.

- Change the invented uplifts to half their size. Does the targeted plan still beat the blanket offer? Which group's offer is the first to stop paying?
- Run the clustering again with `CategoryBreadth` removed and compare the groups. Did the extra column earn its place?

In [ ]:
# Your space.